# 03 — Compact semester tables / big-number brief

## Propósito

Esta notebook convierte la historia temporal ya trabajada por `02_operating_result_report.ipynb` en **tablas compactas**, pensadas para entender números grandes sin perderse en series largas.

La restricción deliberada es fuerte:

- no más de **10 filas** por tabla mostrada;
- no más de **10 columnas** por tabla mostrada;
- preferencia por valores **semestrales** para cubrir todo el período;
- muchas tablas pequeñas, intercaladas con texto explicativo;
- sin gráficos ni series densas: eso queda delegado en Notebook 02.

## Pregunta humana

¿Qué números grandes hay que poder mirar en pocos minutos para entender la escala, los cambios por semestre y el cierre 2026-H1?

## No hace

- No reemplaza la serie temporal de Notebook 02.
- No recalcula contabilidad desde ledger si ya existe el output mensual de Notebook 02.
- No cierra saldo jurídico por actor.
- No convierte funding familiar en ingreso operativo.


## 0. Notebook contract

Este notebook es una **UI de lectura compacta**. Su insumo preferido es:

`out/professional_pack/latest/operating_result/operating_result_monthly.csv`

Ese archivo debería venir de Notebook 02. Si no existe, esta notebook no inventa una historia: deja caveats y muestra qué falta.


In [1]:
from pathlib import Path
import json
import math
import re
from datetime import datetime

import pandas as pd
from IPython.display import display, Markdown, HTML

pd.set_option("display.max_rows", 10)
pd.set_option("display.max_columns", 10)
pd.set_option("display.width", 140)

def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for p in candidates:
        if (p / "accounting").exists() and (p / "out").exists():
            return p
        if (p / "Makefile").exists() and (p / "accounting").exists():
            return p
    return start

REPO = find_repo_root()
OUT = REPO / "out"

METRICS = OUT / "metrics" / "latest"
DEBT = OUT / "debt_resolution" / "latest"
RUN = OUT / "run" / "accounting" / "latest"
OPERATING = OUT / "professional_pack" / "latest" / "operating_result"

PACK = OUT / "professional_pack" / "latest" / "compact_tables"
FIGURES = PACK / "figures"
TABLES = PACK / "tables"
for d in [PACK, FIGURES, TABLES]:
    d.mkdir(parents=True, exist_ok=True)

RUN_LABEL = "latest"
GENERATED_AT = datetime.now().strftime("%Y-%m-%d %H:%M")

caveats = []
qa_rows = []

def note_caveat(area, message, severity="warning"):
    caveats.append({"area": area, "severity": severity, "message": message})

def note_qa(check, status, detail=""):
    qa_rows.append({"check": check, "status": status, "detail": detail})

def read_csv_safe(path, **kwargs):
    path = Path(path)
    if not path.exists():
        note_caveat("missing_input", f"No se encontró {path}", "warning")
        return pd.DataFrame()
    try:
        df = pd.read_csv(path, **kwargs)
        note_qa(f"load:{path.name}", "ok", f"{len(df)} rows, {len(df.columns)} cols")
        return df
    except Exception as exc:
        note_caveat("read_error", f"No se pudo leer {path}: {exc}", "error")
        return pd.DataFrame()

def first_existing(df, names):
    for name in names:
        if name in df.columns:
            return name
    return None

def numeric_series(df, col):
    if col is None or col not in df.columns:
        return pd.Series([pd.NA] * len(df), index=df.index, dtype="Float64")
    return pd.to_numeric(df[col], errors="coerce")

def fmt_money(x):
    if pd.isna(x):
        return "s/d"
    try:
        x = float(x)
    except Exception:
        return "s/d"
    sign = "-" if x < 0 else ""
    return f"{sign}$ {abs(x):,.0f}".replace(",", ".")

def fmt_num(x, digits=1):
    if pd.isna(x):
        return "s/d"
    try:
        return f"{float(x):,.{digits}f}".replace(",", "X").replace(".", ",").replace("X", ".")
    except Exception:
        return "s/d"

def fmt_pct(x):
    if pd.isna(x) or not math.isfinite(float(x)):
        return "s/d"
    return f"{100*float(x):.1f}%".replace(".", ",")

def safe_div(a, b):
    if pd.isna(a) or pd.isna(b) or float(b) == 0:
        return pd.NA
    return float(a) / float(b)

def compact_display(df, name, max_rows=10, max_cols=10):
    """Display and return a compact copy, never more than max_rows x max_cols."""
    if df is None or df.empty:
        note_caveat(name, f"La tabla {name} está vacía.", "warning")
        display(Markdown(f"**{name}:** s/d"))
        return pd.DataFrame()
    out = df.copy()
    original_shape = out.shape
    if out.shape[0] > max_rows:
        note_caveat(name, f"Tabla truncada de {out.shape[0]} a {max_rows} filas.", "info")
        out = out.tail(max_rows)
    if out.shape[1] > max_cols:
        note_caveat(name, f"Tabla truncada de {out.shape[1]} a {max_cols} columnas.", "info")
        out = out.iloc[:, :max_cols]
    note_qa(f"compact_shape:{name}", "ok" if out.shape[0] <= max_rows and out.shape[1] <= max_cols else "fail", f"{original_shape} -> {out.shape}")
    display(out)
    return out

def to_markdown_safe(df):
    if df is None or df.empty:
        return "_s/d_"
    try:
        return df.to_markdown(index=False)
    except Exception:
        return df.to_string(index=False)

display(Markdown(f"**Repo detectado:** `{REPO}`  \
**Salida compacta:** `{PACK}`"))


**Repo detectado:** `/home/matias/repos/accounting-backend`  **Salida compacta:** `/home/matias/repos/accounting-backend/out/professional_pack/latest/compact_tables`

## 1. Cargar outputs canónicos

La tabla compacta consume primero `monthly_operating_statement.csv`, `monthly_cash_close.csv`, `monthly_debt_position.csv` y `frontend_metric_series.csv`. La salida de Notebook 02 queda solo como artefacto de lectura si existe; no se usa para decidir OPEX/caja.


In [2]:
monthly_path = OPERATING / "operating_result_monthly.csv"
regime_path = OPERATING / "regime_periods_used.csv"
semester_close_path = OPERATING / "semester_close_2026H1.csv"

monthly_operating_statement = read_csv_safe(RUN / "monthly_operating_statement.csv")
monthly_cash_close = read_csv_safe(RUN / "monthly_cash_close.csv")
monthly_debt_position = read_csv_safe(RUN / "monthly_debt_position.csv")
frontend_metric_series = read_csv_safe(METRICS / "frontend_metric_series.csv")
metric_contract_frontier = read_csv_safe(METRICS / "metric_contract_frontier.csv")

monthly_raw = read_csv_safe(monthly_path)  # optional notebook-02 output, not semantic source of truth
regime_periods = read_csv_safe(regime_path)
semester_close_existing = read_csv_safe(semester_close_path)

metric_registry = read_csv_safe(METRICS / "metric_registry.csv")
metric_values = read_csv_safe(METRICS / "metric_values.csv")
validation_report = read_csv_safe(METRICS / "validation_report.csv")

display(Markdown("### Inputs cargados"))
input_status = pd.DataFrame([
    {"input": "monthly_operating_statement.csv", "exists": not monthly_operating_statement.empty, "path": str(RUN / "monthly_operating_statement.csv")},
    {"input": "monthly_cash_close.csv", "exists": not monthly_cash_close.empty, "path": str(RUN / "monthly_cash_close.csv")},
    {"input": "monthly_debt_position.csv", "exists": not monthly_debt_position.empty, "path": str(RUN / "monthly_debt_position.csv")},
    {"input": "frontend_metric_series.csv", "exists": not frontend_metric_series.empty, "path": str(METRICS / "frontend_metric_series.csv")},
    {"input": "metric_contract_frontier.csv", "exists": not metric_contract_frontier.empty, "path": str(METRICS / "metric_contract_frontier.csv")},
    {"input": "operating_result_monthly.csv (optional)", "exists": monthly_path.exists(), "path": str(monthly_path)},
])
compact_display(input_status, "input_status", max_rows=10, max_cols=4)


### Inputs cargados

,input,exists,path
0,monthly_operating_statement.csv,True,/home/matias/repos/accounting-backend/out/run/...
1,monthly_cash_close.csv,True,/home/matias/repos/accounting-backend/out/run/...
2,monthly_debt_position.csv,False,/home/matias/repos/accounting-backend/out/run/...
3,frontend_metric_series.csv,True,/home/matias/repos/accounting-backend/out/metr...
4,metric_contract_frontier.csv,True,/home/matias/repos/accounting-backend/out/metr...
5,operating_result_monthly.csv (optional),True,/home/matias/repos/accounting-backend/out/prof...


,input,exists,path
0,monthly_operating_statement.csv,True,/home/matias/repos/accounting-backend/out/run/...
1,monthly_cash_close.csv,True,/home/matias/repos/accounting-backend/out/run/...
2,monthly_debt_position.csv,False,/home/matias/repos/accounting-backend/out/run/...
3,frontend_metric_series.csv,True,/home/matias/repos/accounting-backend/out/metr...
4,metric_contract_frontier.csv,True,/home/matias/repos/accounting-backend/out/metr...
5,operating_result_monthly.csv (optional),True,/home/matias/repos/accounting-backend/out/prof...


## 2. Normalizar espina mensual y semestre

Acá no estamos mostrando la serie larga. Solo la usamos para construir cortes semestrales compactos.


In [3]:
def parse_month_column(df):
    if df.empty:
        return df
    df = df.copy()
    month_col = first_existing(df, ["period_month", "month", "period", "period_id", "date"])
    if month_col is None:
        note_caveat("month_parse", "No se encontró columna mensual.", "warning")
        return df
    s = df[month_col].astype(str)
    dt = pd.to_datetime(s.str.extract(r"(\d{4}-\d{2})")[0], errors="coerce")
    missing = dt.isna()
    if missing.any():
        dt2 = pd.to_datetime(s.where(missing), errors="coerce")
        dt = dt.fillna(dt2)
    df["_month_dt"] = dt
    if "year" not in df.columns:
        df["year"] = df["_month_dt"].dt.year
    if "semester" not in df.columns:
        month_num = df["_month_dt"].dt.month
        sem = month_num.apply(lambda m: "H1" if pd.notna(m) and m <= 6 else ("H2" if pd.notna(m) else pd.NA))
        df["semester"] = df["year"].astype("Int64").astype(str) + sem.astype(str)
        df.loc[df["_month_dt"].isna(), "semester"] = pd.NA
    return df

def statement_line_to_monthly(line, out_col):
    if monthly_operating_statement.empty:
        return pd.DataFrame(columns=["period", out_col])
    sub = monthly_operating_statement[monthly_operating_statement["statement_line"].astype(str).eq(line)].copy()
    if sub.empty:
        note_caveat(out_col, f"No existe statement_line={line} en monthly_operating_statement.csv", "warning")
        return pd.DataFrame(columns=["period", out_col])
    sub[out_col] = pd.to_numeric(sub["amount"], errors="coerce")
    return sub.groupby("period", as_index=False)[out_col].sum(min_count=1)

if monthly_operating_statement.empty:
    normalized = pd.DataFrame()
    note_caveat("canonical_missing", "Falta monthly_operating_statement.csv; no se construye tabla semestral desde fuentes legacy.", "error")
else:
    base = monthly_operating_statement[["period", "period_end", "Currency"]].drop_duplicates().copy()
    # Compact tables currently aggregate currencies only where source already does; keep Currency visible in normalized.
    parts = [
        statement_line_to_monthly("rent_revenue", "rent_total"),
        statement_line_to_monthly("property_opex_true", "opex_total"),
        statement_line_to_monthly("net_operating", "net_operating"),
        statement_line_to_monthly("funding_contributions", "funding_total"),
        statement_line_to_monthly("family_draws_or_distributions", "draws_total"),
    ]
    monthly = base.copy()
    for part in parts:
        if not part.empty:
            monthly = monthly.merge(part, on="period", how="left")
    monthly["period_month"] = monthly["period"]
    normalized = parse_month_column(monthly)
    normalized["regime_label"] = "canonical"

    safe_cash = monthly_cash_close[monthly_cash_close.get("is_frontend_safe", pd.Series(dtype=str)).astype(str).str.lower().isin(["true", "1", "yes", "y"])] if not monthly_cash_close.empty else pd.DataFrame()
    if safe_cash.empty:
        normalized["cash_closing"] = pd.NA
        note_caveat("cash_closing", "No hay filas is_frontend_safe=true en monthly_cash_close.csv; caja queda unavailable.", "warning")
    else:
        safe_cash = safe_cash.copy(); safe_cash["cash_closing"] = pd.to_numeric(safe_cash["close_amount"], errors="coerce")
        cash_m = safe_cash.groupby("period", as_index=False)["cash_closing"].sum(min_count=1)
        normalized = normalized.merge(cash_m, on="period", how="left")

    debt_total = monthly_debt_position[monthly_debt_position.get("component", pd.Series(dtype=str)).astype(str).eq("total")].copy() if not monthly_debt_position.empty else pd.DataFrame()
    if debt_total.empty:
        normalized["debt_closing"] = pd.NA
    else:
        debt_total["debt_closing"] = pd.to_numeric(debt_total.get("open_amount", debt_total.get("open_total")), errors="coerce")
        debt_m = debt_total.groupby("period", as_index=False)["debt_closing"].sum(min_count=1)
        normalized = normalized.merge(debt_m, on="period", how="left")

column_sources = pd.DataFrame([
    {"target": "rent_total", "source": "monthly_operating_statement:rent_revenue"},
    {"target": "opex_total", "source": "monthly_operating_statement:property_opex_true"},
    {"target": "net_operating", "source": "monthly_operating_statement:net_operating"},
    {"target": "funding_total", "source": "monthly_operating_statement:funding_contributions"},
    {"target": "draws_total", "source": "monthly_operating_statement:family_draws_or_distributions"},
    {"target": "cash_closing", "source": "monthly_cash_close:is_frontend_safe=true"},
    {"target": "debt_closing", "source": "monthly_debt_position:component=total"},
])
compact_display(column_sources, "column_sources", max_rows=10, max_cols=2)


,target,source
0,rent_total,monthly_operating_statement:rent_revenue
1,opex_total,monthly_operating_statement:property_opex_true
2,net_operating,monthly_operating_statement:net_operating
3,funding_total,monthly_operating_statement:funding_contributions
4,draws_total,monthly_operating_statement:family_draws_or_di...
5,cash_closing,monthly_cash_close:is_frontend_safe=true
6,debt_closing,monthly_debt_position:component=total


,target,source
0,rent_total,monthly_operating_statement:rent_revenue
1,opex_total,monthly_operating_statement:property_opex_true
2,net_operating,monthly_operating_statement:net_operating
3,funding_total,monthly_operating_statement:funding_contributions
4,draws_total,monthly_operating_statement:family_draws_or_di...
5,cash_closing,monthly_cash_close:is_frontend_safe=true
6,debt_closing,monthly_debt_position:component=total


## 3. Tabla 1 — Mapa semestral compacto

Esta es la tabla principal. La idea es cubrir todo el período en pocos renglones: cada semestre condensa renta, costos, margen, funding, retiros, caja y deuda.

No reemplaza la serie mensual; permite mirar la escala.


In [4]:
def last_non_null(s):
    s = s.dropna()
    return s.iloc[-1] if len(s) else pd.NA

if normalized.empty or normalized["semester"].isna().all():
    semester_overview = pd.DataFrame()
    note_caveat("semester_overview", "No se pudo construir vista semestral: falta espina mensual o columna semester.", "error")
else:
    g = normalized.dropna(subset=["semester"]).groupby("semester", sort=True)
    semester_overview = g.agg(
        months=("semester", "size"),
        regime=("regime_label", lambda s: last_non_null(s.astype(str))),
        rent_total=("rent_total", "sum"),
        opex_total=("opex_total", "sum"),
        net_operating=("net_operating", "sum"),
        funding_total=("funding_total", "sum"),
        draws_total=("draws_total", "sum"),
        cash_end=("cash_closing", last_non_null),
        debt_end=("debt_closing", last_non_null),
    ).reset_index()

    # Keep full 4y/H1 period if <=10; otherwise last 10 by design.
    semester_overview = semester_overview.tail(10)

semester_overview_display = semester_overview.copy()
for c in ["rent_total", "opex_total", "net_operating", "funding_total", "draws_total", "cash_end", "debt_end"]:
    if c in semester_overview_display.columns:
        semester_overview_display[c] = semester_overview_display[c].map(fmt_money)

compact_semester_overview = compact_display(semester_overview_display, "semester_overview", max_rows=10, max_cols=10)


,semester,months,regime,rent_total,opex_total,net_operating,funding_total,draws_total,cash_end,debt_end
0,2022H1,6,canonical,$ 1.694.456,$ 175.100,$ 1.519.356,$ 0,$ 1.519.356,s/d,s/d
1,2022H2,6,canonical,$ 3.254.348,$ 412.166,$ 2.842.182,$ 0,$ 2.842.182,s/d,s/d
2,2023H1,6,canonical,$ 4.007.161,$ 810.256,$ 3.196.905,$ 200.220,$ 3.814.405,s/d,s/d
3,2023H2,7,canonical,$ 5.900.031,$ 3.721.583,$ 2.178.448,$ 1.826.000,$ 5.540.031,s/d,s/d
4,2024H1,10,canonical,$ 9.937.734,$ 4.920.099,$ 5.017.635,$ 1.640.000,$ 9.935.054,s/d,s/d
5,2024H2,12,canonical,$ 15.536.778,$ 11.043.584,$ 4.493.194,$ 4.704.000,$ 15.532.438,s/d,s/d
6,2025H1,12,canonical,$ 25.205.404,$ 10.167.989,$ 15.037.415,$ 4.464.002,$ 22.774.576,s/d,s/d
7,2025H2,12,canonical,$ 55.054.560,$ 14.168.972,$ 40.885.588,$ 2.920.000,$ 46.196.802,s/d,s/d
8,2026H1,12,canonical,$ 59.182.560,$ 14.479.907,$ 44.702.653,$ 220.000,$ 53.032.780,s/d,s/d
9,2026H2,1,canonical,$ 0,$ 182.961,-$ 182.961,$ 0,$ 0,s/d,s/d


## 4. Tabla 2 — Digest de números grandes

Esta tabla baja la ansiedad de los números grandes: para cada bloque muestra el acumulado del período disponible, el promedio por semestre y el cierre / último semestre.


In [5]:
def make_big_number_digest(sem):
    if sem.empty:
        return pd.DataFrame()
    flow_metrics = [
        ("Renta", "rent_total", "Ingreso operativo / renta"),
        ("Costos operativos", "opex_total", "Costos de sostener la operación"),
        ("Resultado operativo", "net_operating", "Renta menos costos, sin funding"),
        ("Funding familiar", "funding_total", "Aportes o financiamiento familiar"),
        ("Retiros / distribución", "draws_total", "Salidas distributivas o retiros"),
    ]
    stock_metrics = [
        ("Caja cierre", "cash_end", "Stock al final del último semestre"),
        ("Deuda interna cierre", "debt_end", "Stock al final del último semestre"),
    ]
    rows = []
    n_sem = max(len(sem), 1)
    latest_sem = sem.iloc[-1] if len(sem) else {}
    latest_label = latest_sem.get("semester", "s/d") if len(sem) else "s/d"

    for label, col, reading in flow_metrics:
        if col not in sem.columns:
            total = pd.NA
            avg = pd.NA
            latest = pd.NA
        else:
            total = pd.to_numeric(sem[col], errors="coerce").sum(skipna=True)
            avg = total / n_sem if n_sem else pd.NA
            latest = latest_sem.get(col, pd.NA)
        rows.append({
            "bloque": label,
            "total_periodo": fmt_money(total),
            "prom_semestre": fmt_money(avg),
            f"{latest_label}": fmt_money(latest),
            "lectura": reading,
        })

    for label, col, reading in stock_metrics:
        latest = latest_sem.get(col, pd.NA) if len(sem) and col in sem.columns else pd.NA
        rows.append({
            "bloque": label,
            "total_periodo": "no aplica",
            "prom_semestre": "no aplica",
            f"{latest_label}": fmt_money(latest),
            "lectura": reading,
        })

    return pd.DataFrame(rows)

big_number_digest = make_big_number_digest(semester_overview)
compact_big_number_digest = compact_display(big_number_digest, "big_number_digest", max_rows=10, max_cols=5)


,bloque,total_periodo,prom_semestre,2026H2,lectura
0,Renta,$ 179.773.032,$ 17.977.303,$ 0,Ingreso operativo / renta
1,Costos operativos,$ 60.082.617,$ 6.008.262,$ 182.961,Costos de sostener la operación
2,Resultado operativo,$ 119.690.415,$ 11.969.041,-$ 182.961,"Renta menos costos, sin funding"
3,Funding familiar,$ 15.974.222,$ 1.597.422,$ 0,Aportes o financiamiento familiar
4,Retiros / distribución,$ 161.187.624,$ 16.118.762,$ 0,Salidas distributivas o retiros
5,Caja cierre,no aplica,no aplica,s/d,Stock al final del último semestre
6,Deuda interna cierre,no aplica,no aplica,s/d,Stock al final del último semestre


## 5. Tabla 3 — Ratios semestrales de lectura

Los ratios son intencionalmente pocos. Sirven para leer relaciones, no para hacer econometría:

- costos sobre renta;
- margen operativo sobre renta;
- funding sobre costos;
- retiros sobre margen operativo.

Cuando el denominador falta o es cero, el ratio queda `s/d`.


In [6]:
def make_ratios(sem):
    if sem.empty:
        return pd.DataFrame()
    rows = []
    for _, r in sem.iterrows():
        rent = r.get("rent_total", pd.NA)
        opex = r.get("opex_total", pd.NA)
        net = r.get("net_operating", pd.NA)
        funding = r.get("funding_total", pd.NA)
        draws = r.get("draws_total", pd.NA)
        rows.append({
            "semester": r.get("semester", "s/d"),
            "opex/renta": fmt_pct(safe_div(opex, rent)),
            "margen/renta": fmt_pct(safe_div(net, rent)),
            "funding/opex": fmt_pct(safe_div(funding, opex)),
            "retiros/margen": fmt_pct(safe_div(draws, net)),
            "cash_end": fmt_money(r.get("cash_end", pd.NA)),
            "debt_end": fmt_money(r.get("debt_end", pd.NA)),
        })
    return pd.DataFrame(rows).tail(10)

semester_ratios = make_ratios(semester_overview)
compact_semester_ratios = compact_display(semester_ratios, "semester_ratios", max_rows=10, max_cols=7)


,semester,opex/renta,margen/renta,funding/opex,retiros/margen,cash_end,debt_end
0,2022H1,"10,3%","89,7%","0,0%","100,0%",s/d,s/d
1,2022H2,"12,7%","87,3%","0,0%","100,0%",s/d,s/d
2,2023H1,"20,2%","79,8%","24,7%","119,3%",s/d,s/d
3,2023H2,"63,1%","36,9%","49,1%","254,3%",s/d,s/d
4,2024H1,"49,5%","50,5%","33,3%","198,0%",s/d,s/d
5,2024H2,"71,1%","28,9%","42,6%","345,7%",s/d,s/d
6,2025H1,"40,3%","59,7%","43,9%","151,5%",s/d,s/d
7,2025H2,"25,7%","74,3%","20,6%","113,0%",s/d,s/d
8,2026H1,"24,5%","75,5%","1,5%","118,6%",s/d,s/d
9,2026H2,s/d,s/d,"0,0%","-0,0%",s/d,s/d


## 6. Tabla 4 — Cierre 2026-H1

Esta tabla toma el semestre de cierre como corte de gobernanza. No resume toda la historia: muestra dónde estamos parados al cierre del primer semestre de 2026.


In [7]:
def make_2026h1_close(sem):
    if sem.empty:
        return pd.DataFrame()
    target = sem[sem["semester"].astype(str).str.upper().eq("2026H1")]
    if target.empty:
        note_caveat("2026H1", "No se encontró semestre 2026H1; se usa el último semestre disponible.", "warning")
        row = sem.iloc[-1]
        sem_label = row.get("semester", "último")
    else:
        row = target.iloc[-1]
        sem_label = "2026H1"

    rows = [
        {"bloque": "Renta", "valor": fmt_money(row.get("rent_total", pd.NA)), "lectura": "Ingreso operativo del semestre"},
        {"bloque": "Costos operativos", "valor": fmt_money(row.get("opex_total", pd.NA)), "lectura": "Costo de sostener el patrimonio"},
        {"bloque": "Resultado operativo", "valor": fmt_money(row.get("net_operating", pd.NA)), "lectura": "Renta menos costos, sin funding"},
        {"bloque": "Funding familiar", "valor": fmt_money(row.get("funding_total", pd.NA)), "lectura": "Aportes o financiamiento para sostener caja"},
        {"bloque": "Retiros / distribución", "valor": fmt_money(row.get("draws_total", pd.NA)), "lectura": "Salidas distributivas del semestre"},
        {"bloque": "Caja cierre", "valor": fmt_money(row.get("cash_end", pd.NA)), "lectura": "Caja visible al final del semestre"},
        {"bloque": "Deuda interna cierre", "valor": fmt_money(row.get("debt_end", pd.NA)), "lectura": "Exposición pendiente al cierre"},
    ]
    out = pd.DataFrame(rows)
    out.insert(0, "semestre", sem_label)
    return out

semester_close_compact = make_2026h1_close(semester_overview)
compact_semester_close = compact_display(semester_close_compact, "semester_close_2026H1_compact", max_rows=10, max_cols=4)


,semestre,bloque,valor,lectura
0,2026H1,Renta,$ 59.182.560,Ingreso operativo del semestre
1,2026H1,Costos operativos,$ 14.479.907,Costo de sostener el patrimonio
2,2026H1,Resultado operativo,$ 44.702.653,"Renta menos costos, sin funding"
3,2026H1,Funding familiar,$ 220.000,Aportes o financiamiento para sostener caja
4,2026H1,Retiros / distribución,$ 53.032.780,Salidas distributivas del semestre
5,2026H1,Caja cierre,s/d,Caja visible al final del semestre
6,2026H1,Deuda interna cierre,s/d,Exposición pendiente al cierre


## 7. Tabla 5 — Claims compactos y soporte

Cada afirmación queda atada a una tabla. Si falta soporte, la afirmación se degrada.


In [8]:
def has_any_nonzero_or_nonmissing(df, col):
    if df.empty or col not in df.columns:
        return False
    s = pd.to_numeric(df[col], errors="coerce")
    return s.notna().any() and (s.fillna(0).abs().sum() != 0)

claim_rows = [
    {
        "claim": "La historia debe leerse por semestres, no solo por total agregado.",
        "support": "semester_overview",
        "status": "ok" if not semester_overview.empty else "missing",
        "risk": "Un total de 4 años mezcla regímenes."
    },
    {
        "claim": "La operación debe separarse del funding familiar.",
        "support": "net_operating + funding_total separados",
        "status": "ok" if ("net_operating" in semester_overview.columns and "funding_total" in semester_overview.columns) else "missing",
        "risk": "Confundir aportes con ingresos operativos."
    },
    {
        "claim": "El cierre 2026-H1 funciona como corte de gobernanza.",
        "support": "semester_close_2026H1_compact",
        "status": "ok" if not semester_close_compact.empty else "missing",
        "risk": "No distinguir historia larga de estado actual."
    },
    {
        "claim": "Los retiros deben compararse contra margen/caja, no contra renta bruta.",
        "support": "semester_ratios",
        "status": "ok" if not semester_ratios.empty else "missing",
        "risk": "Sobreestimar disponibilidad libre."
    },
    {
        "claim": "Deuda y caja son stocks de cierre, no flows.",
        "support": "cash_end / debt_end",
        "status": "ok" if ("cash_end" in semester_overview.columns or "debt_end" in semester_overview.columns) else "missing",
        "risk": "Sumar stocks como si fueran flujos."
    },
]
claims_compact = pd.DataFrame(claim_rows)
compact_claims = compact_display(claims_compact, "claims_compact", max_rows=10, max_cols=4)


,claim,support,status,risk
0,"La historia debe leerse por semestres, no solo...",semester_overview,ok,Un total de 4 años mezcla regímenes.
1,La operación debe separarse del funding familiar.,net_operating + funding_total separados,ok,Confundir aportes con ingresos operativos.
2,El cierre 2026-H1 funciona como corte de gober...,semester_close_2026H1_compact,ok,No distinguir historia larga de estado actual.
3,Los retiros deben compararse contra margen/caj...,semester_ratios,ok,Sobreestimar disponibilidad libre.
4,"Deuda y caja son stocks de cierre, no flows.",cash_end / debt_end,ok,Sumar stocks como si fueran flujos.


## 8. QA y caveats

La salida profesional no debe esconder faltantes. Si una tabla está vacía, truncada o usa fallback, queda registrado.


In [9]:
# Additional QA checks.
note_qa("has_monthly_spine_from_nb02", "ok" if not monthly_raw.empty else "fail", str(monthly_path))
note_qa("has_semester_overview", "ok" if not semester_overview.empty else "fail", f"{semester_overview.shape}")
note_qa("all_display_tables_max_10x10", "ok", "compact_display enforces max 10 rows/cols")
note_qa("delegates_timeseries_to_nb02", "ok", "No charts or long monthly series are displayed here.")
note_qa("has_2026H1_or_latest_close", "ok" if not semester_close_compact.empty else "fail", "")

caveats_df = pd.DataFrame(caveats).drop_duplicates() if caveats else pd.DataFrame(columns=["area", "severity", "message"])
qa_df = pd.DataFrame(qa_rows).drop_duplicates() if qa_rows else pd.DataFrame(columns=["check", "status", "detail"])

display(Markdown("### Caveats"))
compact_caveats = compact_display(caveats_df, "compact_caveats", max_rows=10, max_cols=3)

display(Markdown("### QA"))
compact_qa = compact_display(qa_df, "compact_qa", max_rows=10, max_cols=3)


### Caveats

,area,severity,message
0,missing_input,warning,No se encontró /home/matias/repos/accounting-b...
1,cash_closing,warning,No hay filas is_frontend_safe=true en monthly_...


### QA

,check,status,detail
12,compact_shape:semester_overview,ok,"(10, 10) -> (10, 10)"
13,compact_shape:big_number_digest,ok,"(7, 5) -> (7, 5)"
14,compact_shape:semester_ratios,ok,"(10, 7) -> (10, 7)"
15,compact_shape:semester_close_2026H1_compact,ok,"(7, 4) -> (7, 4)"
16,compact_shape:claims_compact,ok,"(5, 4) -> (5, 4)"
17,has_monthly_spine_from_nb02,ok,/home/matias/repos/accounting-backend/out/prof...
18,has_semester_overview,ok,"(10, 10)"
19,all_display_tables_max_10x10,ok,compact_display enforces max 10 rows/cols
20,delegates_timeseries_to_nb02,ok,No charts or long monthly series are displayed...
21,has_2026H1_or_latest_close,ok,


## 9. Exportar reporte compacto

Exporta tablas y un markdown/html breve. El objetivo es que este reporte pueda insertarse como soporte en briefs, reuniones o revisiones sin arrastrar una serie temporal larga.


In [10]:
# Raw compact outputs.
semester_overview.to_csv(TABLES / "compact_semester_overview_raw.csv", index=False)
semester_overview_display.to_csv(PACK / "compact_semester_overview.csv", index=False)
big_number_digest.to_csv(PACK / "compact_big_number_digest.csv", index=False)
semester_ratios.to_csv(PACK / "compact_semester_ratios.csv", index=False)
semester_close_compact.to_csv(PACK / "compact_semester_close_2026H1.csv", index=False)
claims_compact.to_csv(PACK / "compact_claims.csv", index=False)
caveats_df.to_csv(PACK / "compact_caveats.csv", index=False)
qa_df.to_csv(PACK / "compact_qa.csv", index=False)

report_md = f"""# Tablas compactas — lectura semestral de números grandes

_Generado: {GENERATED_AT}_  
_Período/salida: `{RUN_LABEL}`_

## Lectura

Este reporte no muestra la serie temporal completa. Esa tarea queda en Notebook 02.  
Acá usamos tablas semestrales compactas para entender escala, cambios de régimen y cierre 2026-H1.

## 1. Mapa semestral compacto

{to_markdown_safe(compact_semester_overview)}

## 2. Digest de números grandes

{to_markdown_safe(compact_big_number_digest)}

## 3. Ratios semestrales de lectura

{to_markdown_safe(compact_semester_ratios)}

## 4. Cierre 2026-H1

{to_markdown_safe(compact_semester_close)}

## 5. Claims y soporte

{to_markdown_safe(compact_claims)}

## Caveats

{to_markdown_safe(compact_caveats)}

## QA

{to_markdown_safe(compact_qa)}
"""

(PACK / "compact_tables_report.md").write_text(report_md, encoding="utf-8")

def md_to_basic_html(md):
    # Minimal markdown-ish conversion. Keep this dependency-free.
    import html
    lines = md.splitlines()
    out = [
        "<!doctype html><html lang='es'><head><meta charset='utf-8'>",
        "<title>Tablas compactas</title>",
        "<style>body{font-family:system-ui,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;max-width:980px;margin:40px auto;padding:0 24px 56px;color:#1f2937;line-height:1.45} h1{border-bottom:2px solid #d1d5db;padding-bottom:10px} h2{margin-top:30px} table{border-collapse:collapse;width:100%;font-size:13px;margin:12px 0 24px} th,td{border:1px solid #e5e7eb;padding:6px 8px;text-align:left} th{background:#f3f4f6} code{background:#f3f4f6;padding:2px 5px;border-radius:4px}</style>",
        "</head><body>"
    ]
    in_table = False
    for line in lines:
        if line.startswith("# "):
            out.append(f"<h1>{html.escape(line[2:])}</h1>")
        elif line.startswith("## "):
            out.append(f"<h2>{html.escape(line[3:])}</h2>")
        elif line.startswith("|"):
            # crude passthrough via pandas-generated pipe table to pre for reliability
            out.append(f"<pre>{html.escape(line)}</pre>")
        elif line.strip() == "":
            out.append("")
        else:
            out.append(f"<p>{html.escape(line)}</p>")
    out.append("</body></html>")
    return "\n".join(out)

(PACK / "compact_tables_report.html").write_text(md_to_basic_html(report_md), encoding="utf-8")

display(Markdown(f"""Exportado en:

`{PACK}`

Archivos principales:

- `compact_tables_report.md`
- `compact_tables_report.html`
- `compact_semester_overview.csv`
- `compact_big_number_digest.csv`
- `compact_semester_ratios.csv`
- `compact_semester_close_2026H1.csv`
- `compact_claims.csv`
- `compact_caveats.csv`
- `compact_qa.csv`
"""))


Exportado en:

`/home/matias/repos/accounting-backend/out/professional_pack/latest/compact_tables`

Archivos principales:

- `compact_tables_report.md`
- `compact_tables_report.html`
- `compact_semester_overview.csv`
- `compact_big_number_digest.csv`
- `compact_semester_ratios.csv`
- `compact_semester_close_2026H1.csv`
- `compact_claims.csv`
- `compact_caveats.csv`
- `compact_qa.csv`
